### 미션 1. 1~30위 썸네일 수집하기

- 목표: 다음 버튼을 눌러가며 1~30위 썸네일 URL을 모으기
- 힌트
  - 위 6번(한 페이지 수집)을 3번 반복
  - 매 페이지: `img` 대기 → `plazy` 제외하고 누적
  - 마지막 페이지가 아니면 다음 버튼 클릭 후 `time.sleep(2)`
  - 다음 버튼: `#mor_history_id_0 > div > div.compo-paging > button.btn_next`

In [2]:
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from IPython.display import clear_output

# 앞서 정의한 User-Agent 사용
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36"
url = "https://search.daum.net/search?w=tot&q=2025%EB%85%84%EC%98%81%ED%99%94%EC%88%9C%EC%9C%84&DA=MOR&rtmaxcoll=MOR"

# 드라이버 옵션 설정 및 실행
options = webdriver.ChromeOptions()
options.add_argument("user-agent=" + user_agent)
driver = webdriver.Chrome(options=options)
driver.get(url)

NEXT_BTN = "#mor_history_id_0 > div > div.compo-paging > button.btn_next"
all_image_urls = [] # 수집된 모든 URL을 담을 리스트

# 10개씩 3페이지, 총 3번 반복
for i in range(3):
    # 1. img가 나타날 때까지 대기
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "div.item-thumb c-thumb img"))
    )
    
    # 2. 현재 페이지 파싱
    soup = BeautifulSoup(driver.page_source, "lxml")
    images = soup.select("div > div.item-thumb > c-thumb > div > a > img")
    
    # 3. plazy(지연 로딩) 제외하고 현재 페이지의 썸네일 누적
    page_urls = [img["src"] for img in images if "plazy" not in img["src"]]
    #all_image_urls.extend(page_urls)
    all_image_urls = page_urls

    print(f"\r {i+1}번째 페이지 수집 완료 (현재 누적: {len(all_image_urls)}개)", end="")
    
    # 4. 마지막 페이지(i=2)가 아니면 다음 버튼 클릭 후 대기
    if i < 2:
        driver.find_element(By.CSS_SELECTOR, NEXT_BTN).click()
        time.sleep(2)   # 새 썸네일 채워질 시간

# 브라우저 종료
driver.quit()

# 최종 결과 확인
clear_output()
print("전체 수집된 썸네일 수:", len(all_image_urls))

전체 수집된 썸네일 수: 30


### 미션 2. 수집한 썸네일을 파일로 저장하기

- 목표: 미션 1에서 모은 결과를 이미지 파일로 저장
- 힌트
  - 1일차에 배운 `requests.get` + 파일 쓰기(`wb`)
  - 파일명 예: `movie_images/2025_movie_rank_1.jpg`

In [3]:
import os
import requests

folder_name = "movie_images"

if not os.path.exists(folder_name):
    os.makedirs(folder_name)

print("썸네일 이미지 다운로드를 시작합니다...")

# 1. 리스트에서 중복 URL 제거 (순서는 그대로 유지)
unique_urls = []
for url in all_image_urls:
    if url not in unique_urls:
        unique_urls.append(url)

# 2. 딱 30개까지만 자르기
final_urls = unique_urls[:30]

# 3. 정리된 final_urls로 저장 진행
for idx, img_url in enumerate(final_urls, start=1):
    response = requests.get(img_url, headers={"User-Agent": user_agent})
    response.raise_for_status() # 에러 탈출을 위한 코드
    
    file_path = f"{folder_name}/2025_movie_rank_{idx}.jpg" # 파일 저장하기
    
    with open(file_path, "wb") as f:
        f.write(response.content) 
        
    print(f"\r[{idx}/30] 저장 완료: {file_path}", end="")

clear_output()
print("30위까지 썸네일 이미지 저장이 완료되었습니다!")

30위까지 썸네일 이미지 저장이 완료되었습니다!



## 8. 실전 3: 다음 뉴스 (탭 클릭 후 헤드라인 수집)

- 카테고리 탭을 클릭하며 헤드라인 제목/링크 수집
- http://news.daum.net

### 미션 3. 탭을 순회하며 제목과 링크 수집하기

- 목표: 카테고리 탭(사회·경제·정치 등)을 클릭하며 각 헤드라인의 제목과 링크 모으기
- 힌트
  - 탭 클릭: `driver.find_element(By.LINK_TEXT, 탭이름).click()`
  - 헤드라인 목록: `ul.list_newsheadline2` 안의 `li`들
  - 제목: `strong.tit_txt`의 텍스트 / 링크: `a`의 `href`
  - 결과를 `pd.DataFrame(data, columns=["category","title","link"])`로

In [6]:
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 1. 준비 및 브라우저 실행
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
options = webdriver.ChromeOptions()
options.add_argument("user-agent=" + user_agent)

driver = webdriver.Chrome(options=options)

url = "https://www.aladin.co.kr/shop/common/wbest.aspx?BestType=Bestseller&BranchType=1&CID=1108"
driver.get(url)

aladin_image_urls = []

try:
    # 2. 제공해주신 Xpath를 CSS 셀렉터 형태로 변환하여 대기 로직에 적용
    # #Myform 내부에 도서 테이블tr들의 공통적인 이미지 경로를 대기합니다.
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "#Myform table tr div a div div img"))
    )
    time.sleep(2)  # 이미지가 완전히 로드될 때까지 2초간 안전하게 대기
    
    # 3. 현재 페이지 HTML 파싱
    soup = BeautifulSoup(driver.page_source, "lxml")
    
    # 주신 Xpath 구조를 기반으로 모든 책의 썸네일을 집어올 수 있는 CSS Selector 구조
    # tr/td[2] 부분의 가변적인 tr들을 전부 찾기 위해 공통 구조를 엮었습니다.
    book_images = soup.select("#Myform div table tr div > a > div > div > img")
    
    # 4. 이미지 태그에서 src 주소만 추출하기
    for img in book_images:
        if "src" in img.attrs:
            img_url = img["src"]
            
            # 알라딘의 경우 가짜 배너/아이콘을 제외하기 위해 cover 혹은 i_list가 포함된 것 위주로 정제
            if "cover" in img_url or "i_list" in img_url:
                # 중복 수집 방지
                if img_url not in aladin_image_urls:
                    aladin_image_urls.append(img_url)

    print(f"현재 페이지에서 총 {len(aladin_image_urls)}개의 도서 썸네일 URL을 수집했습니다.")

except Exception as e:
    print(f"크롤링 중 에러 발생: {e}")

# 브라우저 종료
driver.quit()

# 5. 수집된 결과 전체 출력 확인
print("=" * 30)
for idx, img_url in enumerate(aladin_image_urls, start=1):
    print(f"[{idx}] {img_url}")

현재 페이지에서 총 50개의 도서 썸네일 URL을 수집했습니다.
[1] https://image.aladin.co.kr/product/39408/42/cover200/k772139099_1.jpg
[2] https://image.aladin.co.kr/product/39176/41/cover200/k882138678_2.jpg
[3] https://image.aladin.co.kr/product/39580/77/cover200/k082139246_1.jpg
[4] https://image.aladin.co.kr/product/39134/78/cover200/k982137650_1.jpg
[5] https://image.aladin.co.kr/product/39375/36/cover200/8936450530_1.jpg
[6] https://image.aladin.co.kr/product/39137/96/cover200/k162138768_3.jpg
[7] https://image.aladin.co.kr/product/28529/48/cover200/8969524894_1.jpg
[8] https://image.aladin.co.kr/product/39446/25/cover200/k532139495_1.jpg
[9] https://image.aladin.co.kr/product/39448/53/cover200/k752139497_1.jpg
[10] https://image.aladin.co.kr/product/39138/67/cover200/k682138769_2.jpg
[11] https://image.aladin.co.kr/product/39523/72/cover200/k372139617_2.jpg
[12] https://image.aladin.co.kr/product/39544/43/cover200/k492139842_2.jpg
[13] https://image.aladin.co.kr/product/39469/93/cover200/k732139217_2.jp

In [7]:
import os
import requests
from IPython.display import clear_output  # clear_output을 사용하기 위한 모듈

# 1. 저장할 폴더 이름 설정 (알라딘 도서용)
folder_name = "aladin_books"
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"

# 폴더가 없으면 생성
if not os.path.exists(folder_name):
    os.makedirs(folder_name)

print("도서 썸네일 이미지 다운로드를 시작합니다...")

# 2. 리스트에서 중복 URL 제거 (이전 단계에서 수집한 aladin_image_urls 사용)
unique_urls = []
for url in aladin_image_urls:
    if url not in unique_urls:
        unique_urls.append(url)

# 수집된 전체 개수 확인 (따로 30개로 자르지 않고 수집된 전체를 저장)
final_urls = unique_urls
total_count = len(final_urls)

# 3. 이미지 다운로드 및 파일 저장 진행
for idx, img_url in enumerate(final_urls, start=1):
    try:
        # 혹시 모를 멈춤 방지를 위해 timeout=5 옵션 추가
        response = requests.get(img_url, headers={"User-Agent": user_agent}, timeout=5)
        response.raise_for_status() # 정상 응답 확인
        
        # 파일명 변경 (예: aladin_books/aladin_bestseller_1.jpg)
        file_path = f"{folder_name}/aladin_bestseller_{idx}.jpg" 
        
        with open(file_path, "wb") as f:
            f.write(response.content) 
            
        # \r을 통해 한 줄에서 덮어쓰며 진행 상황 출력
        print(f"\r[{idx}/{total_count}] 저장 완료: {file_path}", end="")
        
    except Exception as e:
        print(f"\n[{idx}/{total_count}] 저장 실패: {e}")

# 완료 후 이전 출력 내용 지우기 및 최종 메시지
clear_output()
print("=" * 30)
print(f"총 {total_count}개의 도서 썸네일 이미지 저장이 완벽하게 완료되었습니다! 🎉")

총 50개의 도서 썸네일 이미지 저장이 완벽하게 완료되었습니다! 🎉


In [4]:
import pandas as pd

# 1. 준비 및 브라우저 실행
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
options = webdriver.ChromeOptions()
options.add_argument("user-agent=" + user_agent)

driver = webdriver.Chrome(options=options)
driver.get("http://news.daum.net")

# 수집할 카테고리 탭 이름들
categories = ["사회", "정치", "경제", "국제", "문화", "IT/과학"]
data = [] # 결과를 모을 빈 리스트

# 카테고리 탭 전환
for cat in categories:
    try:
        # 해당 카테고리 탭 클릭
        driver.find_element(By.LINK_TEXT, cat).click()
        
        # 헤드라인 목록(ul.list_newsheadline2)이 나타날 때까지 대기
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "ul.list_newsheadline2"))
        )
        time.sleep(1)
        
        
        soup = BeautifulSoup(driver.page_source, "lxml")
        
        # 헤드라인 안의 li 요소들 모두 찾기
        news_list = soup.select("ul.list_newsheadline2 > li")
        
        # 각 뉴스 항목별로 텍스트와 링크 추출
        for news in news_list:
            # 링크는 'a' 태그의 'href' 속성
            a_tag = news.select_one("a")
            link = a_tag["href"] if a_tag else ""
            
            # 'strong.tit_txt'의 텍스트
            title_tag = news.select_one("strong.tit_txt")
            title = title_tag.text.strip() if title_tag else ""
            
            # 제목과 링크 data 리스트에 추가
            if title and link:
                data.append([cat, title, link])
                
        print(f"\r[{cat}] 기사 수집 완료", end="")
        
    except Exception as e:
        print(f"\r[{cat}] 탭 수집 중 에러 발생: {e}", end="")

# 브라우저 종료
driver.quit()

# 3. 판다스 데이터프레임으로 변환
df = pd.DataFrame(data, columns=["category", "title", "link"])

clear_output()
print("총 수집된 뉴스 기사 수:", len(df))

# 데이터프레임 상위 10개 출력 확인
df.head(10)

총 수집된 뉴스 기사 수: 59


,category,title,link
0,사회,"‘죄송합니다, 잘못했습니다’…숨진 교사들 유서에 반복된 말 [플러스 인터뷰]",https://v.daum.net/v/20260620140220127
1,사회,[이통장 발언대] 40년 식당 운영하며 주민 화합 이끈 김영희 이장…“진남리는 함께...,https://v.daum.net/v/20260620135539022
2,사회,"조선일보 ""대통령과 친명의 공격, 윤석열 당무개입 닮아""",https://v.daum.net/v/20260620135302002
3,사회,"""안효섭·전지현 드라마 못 봐요?""…JTBC 사태에 업계 '술렁' [김소연의 엔터비즈]",https://v.daum.net/v/20260620132302694
4,사회,[영상] “어디 경찰이라구요?” 손님 전화 뺏어 받은 택시기사,https://v.daum.net/v/20260620120118772
5,사회,유럽 또 최악의 폭염‥6월인데 40도 돌파. 유럽은 왜 이런가? [기후인사이트 34...,https://v.daum.net/v/20260620114009616
6,사회,폭행·협박 있는 강간 단 7%…‘비동의 간음죄’ 이번엔 통과될까,https://v.daum.net/v/20260620113217544
7,사회,행안부가 왜?…탈모 건보 논의 둘러싼 3가지 궁금증,https://v.daum.net/v/20260620111719298
8,사회,“쌍둥이 득표로 끝까지 싸울순 없다”…‘필즈상’ 허준이 부친의 묘안,https://v.daum.net/v/20260620100225329
9,정치,"국민의힘 ""연어술파티 거짓선동 확인"" 민주 ""판결 본질 왜곡""",https://v.daum.net/v/20260620143942611
